# PARADIS — Reviewer demo

This notebook runs a complete dispersal simulation for the Black-winged Kite
(*Elanus caeruleus*) over France, using known dispersal parameters, at 1 km
resolution on a 200 x 200 km grid. It mirrors
`examples/quick_start_run_knowing_params.py`.

Everything needed (package, example data) is already installed in this
container — just run all cells (`Run` -> `Run All Cells`).

In [ ]:
%matplotlib inline

import pathlib
import torch

from paradis.simulation import PopulationSimulator, show_expansion, show_final
from paradis.io import load_hs, load_obs, load_mask

print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 1. Load rasters

NODATA handling, scaling, and normalisation are handled automatically by
`paradis.io`.

In [ ]:
_DATA = pathlib.Path("examples") / "data"

HS_PATH = _DATA / "HS_Elanus_caeruleus_200km.tif"
MASK_PATH = _DATA / "depFrance_200km.tif"
INIT_PATH = _DATA / "Init_distrib_Elanus_caeruleus_200km_sim.tif"

hs = load_hs(HS_PATH)
france_mask = load_mask(MASK_PATH)
init_distrib = load_obs(INIT_PATH)

hs *= france_mask

print(f"Map shape   : {hs.shape}")
print(f"HS range    : {hs[hs > 0].min():.4f} - {hs.max():.4f}")
print(f"Init patch  : {(init_distrib > 0).sum()} px  (value {init_distrib.max():.4f})")

## 2. Set up the simulator

Parameters below were estimated with the calibration pipeline
(`examples/quick_start_learning_params.py`) and are hard-coded here for
convenience.

In [ ]:
PRESENCE_THRESHOLD = 0.017

sim = PopulationSimulator(
    hs=hs,
    ewalk=924.5,
    n=1980.5,
    r=0.000918,
    tgrowth=2.509,
    carrying_capacity_params=(0.02422, -7.2439, 0.5344),
    solver="sparse",
    presence_threshold=PRESENCE_THRESHOLD,
)

print("Simulator ready:", sim)

## 3. Run the simulation

In [ ]:
N_STEPS = 15

sim.run(
    n_steps=N_STEPS,
    init_distrib=init_distrib,
    window_size=60,
    sub_window=30,
)

## 4. Visualise

In [ ]:
show_expansion(sim)
show_final(sim)

## 5. (Optional) run the test suite

Open a terminal in Jupyter Lab (`File` -> `New` -> `Terminal`) and run:

```bash
pytest
```